In [2]:
# ══════════════════════════════════════════════════════════════════════
# TWO-STREAM v2  —  higher resolution, cosine+warmup, EMA, 8-way TTA
#   MODE = "cv"       -> patient-grouped 5-fold      (role_f0..4)
#   MODE = "official" -> official TCIA split         (role_of0..4)
#   Architecture UNCHANGED: mask-weighted pooling + wide stream + aux heads.
#   Every (fold, seed) checkpointed to .npz - a crash costs one fold.
#   >>> RUN WITH QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time, math, json
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ─────────────────────────── CONFIG ───────────────────────────
QUICK_TEST = False          # <<< True first. One fold, 3 epochs, error check only.
MODE       = "cv"          # "cv" or "official"
TAG        = "twostream_v2"

ST, SW     = 640, 448      # was 512 / 384   <- main gain. Drop to 576/416 if OOM.
BATCH, ACC = 6, 2          # effective batch 12. Lower BATCH if OOM.
SEEDS      = [11, 22]      # [11] for a single-seed run
EPOCHS, FREEZE, WARMUP = 26, 3, 2
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 8e-5    # LR_BACK was 3e-5 - too low
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 8, 2.0
EMA_DECAY  = 0.999
TTA_N      = 8             # was 4
FOLDS      = [0, 1, 2, 3, 4]
USE_AMP    = True
# ──────────────────────────────────────────────────────────────

if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, WARMUP, MULT, FOLDS = [11], 3, 1, 1, 1, [0]
    print(">>> QUICK TEST: 1 fold, 3 epochs, no checkpoint written\n")

ROLE = "role_f" if MODE == "cv" else "role_of"
CKPT = os.path.join(D, "ckpt_%s_%s" % (TAG, MODE)); os.makedirs(CKPT, exist_ok=True)

WIDE = os.path.join(D, "crops_wide_%s_official" % LES)
if not os.path.isdir(WIDE): WIDE = os.path.join(D, "crops_wide_%s" % LES)
PM   = os.path.join(D, "predmasks_%s_official" % LES)
if not os.path.isdir(PM):   PM = os.path.join(D, "predmasks_%s" % LES)
print("wide crops : %s\npred masks : %s" % (WIDE, PM))

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png", ""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s + "_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing"
assert "%s0" % ROLE in d.columns, "%s0 column missing" % ROLE

if MODE == "official":
    _te = d["official_split"].astype(str).str.lower().str.contains("test")
    for _k in range(5):
        assert ((d["%s%d" % (ROLE, _k)] == "test") == _te).all(), "fold %d != official test" % _k
    print("verified OFFICIAL SPLIT | train %d | test %d" % ((~_te).sum(), _te.sum()))
else:
    print("patient-grouped 5-fold CV | %d regions | %d patients"
          % (len(d), d.patient_id.nunique()))
print("malignant %.1f%%\n" % (100 * d.label.mean()))

# ─────────────────── cache (RAM ~2 GB at 640/448) ───────────────────
cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size, size), np.uint8)
    if im.shape != (size, size):
        im = cv2.resize(im, (size, size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im > 127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d crops in %.0fs (~%.1f GB)"
      % (len(CACHE), time.time() - t0, len(CACHE) * (2*ST*ST + 2*SW*SW) / 1e9))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"]); mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta, "\n")

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug = np.asarray(idx), aug
        self.mult, self.tta = (mult if aug else 1), tta
    def __len__(self): return len(self.idx) * self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand() < .5, np.random.rand() < .5
            kk = np.random.randint(4)
            aff = np.random.rand() < .7
            ang, sc = np.random.uniform(-25, 25), np.random.uniform(.88, 1.14)
            itn = np.random.rand() < .5
            gg, bb = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
            def T(im, mk, s):
                if fh: im, mk = im[:, ::-1], mk[:, ::-1]
                if fv: im, mk = im[::-1, :], mk[::-1, :]
                if kk: im, mk = np.rot90(im, kk), np.rot90(mk, kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2, s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s, s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s, s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32) * gg + bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta; rot, flip = t % 4, t // 4      # 8 dihedral transforms
            def V(im, mk):
                if rot: im, mk = np.rot90(im, rot), np.rot90(mk, rot)
                if flip: im, mk = im[:, ::-1], mk[:, ::-1]
                return np.ascontiguousarray(im), np.ascontiguousarray(mk)
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti, tm), (wi, wm)]:
            im = np.ascontiguousarray(im).astype(np.float32) / 255.0
            out.append(torch.from_numpy(((np.stack([im]*3, 0) - MEAN) / STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64)
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024 * 4
        self.head = nn.Sequential(nn.Linear(Fd, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd, 128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * mm
        return torch.cat([(f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6), f.mean((2, 3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone().float()
    def state(self, model):
        ref = model.state_dict()
        return {k: self.shadow[k].to(ref[k].dtype) for k in ref}

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, n_tta=1):
    net.eval(); tot = None
    for t in range(n_tta):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=max(4, BATCH),
                        shuffle=False, num_workers=0, pin_memory=True)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / n_tta

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr] == 0).sum()), float((yy[tr] == 1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters() if not (n_.startswith("bt.") or n_.startswith("bw."))]

    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD)
    sch, ema = None, None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad, skipped = -1.0, None, 0, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            T = max(1, EPOCHS - FREEZE)
            def lam(e):
                if e < WARMUP: return (e + 1) / WARMUP
                return 0.5 * (1 + math.cos(math.pi * (e - WARMUP) / max(1, T - WARMUP)))
            sch = torch.optim.lr_scheduler.LambdaLR(opt, lam)
            ema = EMA(net, EMA_DECAY)

        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        opt.zero_grad(set_to_none=True)
        for bi, (xt, mt, xw, mw, t_, a_) in enumerate(tl):
            xt, mt, xw, mw = (xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True),
                              xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True))
            t_, a_ = t_.to(DEV), a_.to(DEV)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(gg.float(), a_[:, h], ignore_index=-1)
                        for h, gg in enumerate(ax)) / len(ax)
                loss = loss / ACC
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward()
            if (bi + 1) % ACC == 0:
                scaler.unscale_(opt)
                gn = torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                if torch.isfinite(gn): scaler.step(opt)
                else: skipped += 1
                scaler.update(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(net)
        if sch is not None: sch.step()

        # evaluate the EMA weights once they exist
        if ema is not None:
            raw = {k: v.detach().clone() for k, v in net.state_dict().items()}
            net.load_state_dict(ema.state(net))
        pv = predict(net, va, n_tta=1)
        if not np.all(np.isfinite(pv)):
            print("      ep %2d  NON-FINITE predictions - diverged" % ep)
            if bstate is not None: break
            raise RuntimeError("diverged; set USE_AMP = False and rerun")
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}; star = " *"
        else: bad += 1
        if ema is not None: net.load_state_dict(raw)
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break

    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, n_tta=TTA_N)
    if skipped: print("      (%d optimiser steps skipped on non-finite grads)" % skipped)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

# ─────────────────────────── run ───────────────────────────
y = d["label"].values
acc_p, cnt_p = np.zeros(len(d)), np.zeros(len(d))
oof = np.full(len(d), np.nan)

for k in FOLDS:
    role = d["%s%d" % (ROLE, k)]
    tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    ps, t0 = [], time.time()
    for sd in SEEDS:
        ck = os.path.join(CKPT, "f%d_s%d.npz" % (k, sd))
        if os.path.exists(ck) and not QUICK_TEST:
            z = np.load(ck, allow_pickle=True)
            if list(z["te_idx"]) == list(te):
                ps.append(z["prob"]); print("    seed %d: loaded checkpoint (val %.4f)" % (sd, float(z["val"])))
                continue
            print("    seed %d: stale checkpoint, retraining" % sd)
        p_, bv = train_one(tr, va, te, sd); ps.append(p_)
        if not QUICK_TEST:
            np.savez(ck, prob=p_, val=bv, te_idx=np.array(te))
        print("    seed %d: best val %.4f | fold test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p_)))
    pmn = np.mean(ps, 0)
    acc_p[te] += pmn; cnt_p[te] += 1; oof[te] = acc_p[te] / cnt_p[te]
    print("  FOLD %d AUC %.4f | pooled so far %.4f (%.0fs)"
          % (k, roc_auc_score(y[te], pmn), roc_auc_score(y[te], oof[te]), time.time() - t0))

done = ~np.isnan(oof)
res = d.loc[done, ["img", "lesion_key", "label"]].copy(); res["prob"] = oof[done]
Lg = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
print("\n" + "=" * 70)
print("TWO-STREAM v2   MODE=%s   ST=%d SW=%d   seeds=%s" % (MODE, ST, SW, SEEDS))
print("=" * 70)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(res.label, res.prob), len(res)))
print("  per-lesion AUC %.4f  (n=%d)" % (roc_auc_score(Lg.y, Lg.p), len(Lg)))
print("  reference: v1 official image 0.8769 / lesion 0.9043 | v1 CV image 0.8734 / lesion 0.8885")
if not QUICK_TEST:
    suffix = "officialsplit" if MODE == "official" else "cv"
    out = os.path.join(D, "cv_%s_%s_%s_oof.csv" % (LES, TAG, suffix))
    res.rename(columns={"label": "true"}).to_csv(out, index=False)
    print("\n  saved %s" % os.path.basename(out))
else:
    print("\n  QUICK TEST clean. Set QUICK_TEST = False and rerun.")

wide crops : /root/autodl-tmp/CBIS/crops_wide_mass_official
pred masks : /root/autodl-tmp/CBIS/predmasks_mass_official
patient-grouped 5-fold CV | 1696 regions | 892 patients
malignant 46.2%

cached 1696 crops in 20s (~2.1 GB)
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5} 


### fold 0 | train 1054 val 264 test 378
      ep  1  val-AUC 0.7412 *
      ep  2  val-AUC 0.7703 *
      ep  3  val-AUC 0.7283
      ep  4  val-AUC 0.7070
      ep  5  val-AUC 0.7485
      ep  6  val-AUC 0.7909 *
      ep  7  val-AUC 0.8279 *
      ep  8  val-AUC 0.8586 *
      ep  9  val-AUC 0.8797 *
      ep 10  val-AUC 0.8857 *
      ep 11  val-AUC 0.8916 *
      ep 12  val-AUC 0.9007 *
      ep 13  val-AUC 0.9056 *
      ep 14  val-AUC 0.9080 *
      ep 15  val-AUC 0.9084 *
      ep 16  val-AUC 0.9061
      ep 17  val-AUC 0.9002
      ep 18  val-AUC 0.8965
      ep 19  val-AUC 0.8942
      ep 20  val-AUC 0.8912
      ep 21  val-AUC 0.8883
      ep 22  val-AUC 0.8861
      ep 23  val-AUC 0.

In [1]:
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int); d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
B=d.set_index("_k")

for fn in ("cv_mass_twostream_officialsplit_oof.csv",
           "cv_mass_twostream_officialsplit_oof_cvmasks.csv"):
    f=os.path.join(D,fn)
    if not os.path.exists(f): print("MISSING %s"%fn); continue
    m=pd.read_csv(f); m["_k"]=m["img"].map(stem); m=m[m["_k"].isin(B.index)]
    row=[fn.replace("cv_mass_twostream_officialsplit_oof","").replace(".csv","") or "  (standard)"]
    for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
        t=pd.DataFrame(dict(p=m.prob.values,y=B.loc[m._k,"y"].values,
                            k=(m._k.values if key is None else B.loc[m._k,key].values)))
        g=t.groupby("k").agg(p=("p","mean"),y=("y","max"))
        row.append("%s %.4f"%(nm,roc_auc_score(g.y,g.p)))
    print("  ".join(row))
print("\nIf the two lines are close, mask provenance is not a problem.")
print("If _cvmasks is clearly lower, the standard file used masks from a segmentation")
print("model that saw the official-test patients, and _cvmasks is what you must report.")

  (standard)  ROI 0.8769  LESION 0.9043  BREAST 0.9016  PATIENT 0.9041
_cvmasks  ROI 0.8695  LESION 0.9004  BREAST 0.8948  PATIENT 0.8967

If the two lines are close, mask provenance is not a problem.
If _cvmasks is clearly lower, the standard file used masks from a segmentation
model that saw the official-test patients, and _cvmasks is what you must report.
